In [20]:
!pip install torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 26.8 MB/s eta 0:00:00


In [1]:
# import
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import numpy as numpy
import os
from PIL import Image

from tqdm import tqdm
import wandb
wandb.login()


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: km936 (km936-cornell-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

Load the Dataset

In [2]:
!wget -c http://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_train_HR.zip

--2026-04-24 18:22:01--  http://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_train_HR.zip
Resolving data.vision.ee.ethz.ch (data.vision.ee.ethz.ch)... 129.132.52.178, 2001:67c:10ec:36c2::178
Connecting to data.vision.ee.ethz.ch (data.vision.ee.ethz.ch)|129.132.52.178|:80... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_train_HR.zip [following]
--2026-04-24 18:22:01--  https://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_train_HR.zip
Connecting to data.vision.ee.ethz.ch (data.vision.ee.ethz.ch)|129.132.52.178|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3530603713 (3.3G) [application/zip]
Saving to: ‘DIV2K_train_HR.zip’

DIV2K_train_HR.zip  100%[===================>]   3.29G  22.6MB/s    in 2m 28s  

2026-04-24 18:24:30 (22.7 MB/s) - ‘DIV2K_train_HR.zip’ saved [3530603713/3530603713]



In [3]:
!unzip DIV2K_train_HR.zip

Archive:  DIV2K_train_HR.zip
   creating: DIV2K_train_HR/
  inflating: DIV2K_train_HR/0103.png  
  inflating: DIV2K_train_HR/0413.png  
  inflating: DIV2K_train_HR/0031.png  
  inflating: DIV2K_train_HR/0660.png  
  inflating: DIV2K_train_HR/0126.png  
  inflating: DIV2K_train_HR/0793.png  
  inflating: DIV2K_train_HR/0764.png  
  inflating: DIV2K_train_HR/0550.png  
  inflating: DIV2K_train_HR/0437.png  
  inflating: DIV2K_train_HR/0374.png  
  inflating: DIV2K_train_HR/0755.png  
  inflating: DIV2K_train_HR/0614.png  
  inflating: DIV2K_train_HR/0646.png  
  inflating: DIV2K_train_HR/0371.png  
  inflating: DIV2K_train_HR/0312.png  
  inflating: DIV2K_train_HR/0108.png  
  inflating: DIV2K_train_HR/0556.png  
  inflating: DIV2K_train_HR/0794.png  
  inflating: DIV2K_train_HR/0722.png  
  inflating: DIV2K_train_HR/0780.png  
  inflating: DIV2K_train_HR/0555.png  
  inflating: DIV2K_train_HR/0439.png  
  inflating: DIV2K_train_HR/0396.png  
  inflating: DIV2K_train_HR/0666.png  
  infl

In [4]:
!rm -r DIV2K_train_HR.zip

In [5]:
!wget -c http://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_valid_HR.zip

--2026-04-24 18:25:04--  http://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_valid_HR.zip
Resolving data.vision.ee.ethz.ch (data.vision.ee.ethz.ch)... 129.132.52.178, 2001:67c:10ec:36c2::178
Connecting to data.vision.ee.ethz.ch (data.vision.ee.ethz.ch)|129.132.52.178|:80... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_valid_HR.zip [following]
--2026-04-24 18:25:05--  https://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_valid_HR.zip
Connecting to data.vision.ee.ethz.ch (data.vision.ee.ethz.ch)|129.132.52.178|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 448993893 (428M) [application/zip]
Saving to: ‘DIV2K_valid_HR.zip’

DIV2K_valid_HR.zip  100%[===================>] 428.19M  23.1MB/s    in 20s     

2026-04-24 18:25:25 (21.9 MB/s) - ‘DIV2K_valid_HR.zip’ saved [448993893/448993893]



In [6]:
!unzip DIV2K_valid_HR.zip

Archive:  DIV2K_valid_HR.zip
   creating: DIV2K_valid_HR/
  inflating: DIV2K_valid_HR/0897.png  
  inflating: DIV2K_valid_HR/0887.png  
  inflating: DIV2K_valid_HR/0806.png  
  inflating: DIV2K_valid_HR/0834.png  
  inflating: DIV2K_valid_HR/0896.png  
  inflating: DIV2K_valid_HR/0881.png  
  inflating: DIV2K_valid_HR/0828.png  
  inflating: DIV2K_valid_HR/0833.png  
  inflating: DIV2K_valid_HR/0877.png  
  inflating: DIV2K_valid_HR/0826.png  
  inflating: DIV2K_valid_HR/0879.png  
  inflating: DIV2K_valid_HR/0812.png  
  inflating: DIV2K_valid_HR/0809.png  
  inflating: DIV2K_valid_HR/0865.png  
  inflating: DIV2K_valid_HR/0882.png  
  inflating: DIV2K_valid_HR/0830.png  
  inflating: DIV2K_valid_HR/0892.png  
  inflating: DIV2K_valid_HR/0859.png  
  inflating: DIV2K_valid_HR/0858.png  
  inflating: DIV2K_valid_HR/0816.png  
  inflating: DIV2K_valid_HR/0836.png  
  inflating: DIV2K_valid_HR/0857.png  
  inflating: DIV2K_valid_HR/0824.png  
  inflating: DIV2K_valid_HR/0823.png  
  infl

In [7]:
!rm -r DIV2K_valid_HR.zip

Dataset Settings

In [8]:
# check where dataset is loaded relative to colab files
root_path_to_image_train = '/content/DIV2K_train_HR'
root_path_to_image_valid = '/content/DIV2K_valid_HR'

# set the batch size and workers here
batch_size = 16
num_workers = 0

In [9]:
# Set up dataset

class Div2KDataset(Dataset):
    def __init__(self, root, transforms=None):
        self.path_to_img = []
        for f in os.listdir(root):
            if f.endswith('png'):
                self.path_to_img.append(os.path.join(root, f))

        self.transforms = transforms

    def __len__(self):
        return len(self.path_to_img)

    def __getitem__(self, idx):
        image = Image.open(self.path_to_img[idx]).convert('RGB')
        if self.transforms:
            image = self.transforms(image)
        return image

In [10]:
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(360, pad_if_needed=True),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

validation_transform = transforms.Compose([
    transforms.CenterCrop(360), # keep crop deterministic
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]) # same normalization as train
])

train_dataset = Div2KDataset(root_path_to_image_train, transforms=train_transform)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)

valid_dataset = Div2KDataset(root_path_to_image_valid, transforms=validation_transform)
valid_loader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)

In [11]:
# Check dataset sizes
print(f"Train samples: {len(train_dataset)}")   # 800
print(f"Valid samples: {len(valid_dataset)}")   # 100

# Check a single image loading correctly
sample = train_dataset[0]
print(f"Sample type: {type(sample)}")
print(f"Sample shape: {sample.shape}") # [3, 360, 360]

# Check dataloaders
train_batch = next(iter(train_loader))
print(f"Train batch shape: {train_batch.shape}")  # [batch_size, 3, 360, 360]

valid_batch = next(iter(valid_loader))
print(f"Valid batch shape: {valid_batch.shape}") # [batch_size, 3, 360, 360]

Train samples: 800
Valid samples: 100
Sample type: <class 'torch.Tensor'>
Sample shape: torch.Size([3, 360, 360])
Train batch shape: torch.Size([16, 3, 360, 360])
Valid batch shape: torch.Size([16, 3, 360, 360])


Evaluation Functions

In [21]:
import torch
import torch.nn.functional as F
from torchmetrics.image import StructuralSimilarityIndexMeasure

def rs_bpp(D, message, decoded_message):
  """
  Inputs:
  - D: how many bits you store per pixel
  - message: the original random message N x D x H x W (binary 0s and 1s)
  - decoded_message: raw decoder output N x D x H x W (floats/logits)

  Returns: scalar RS-BPP value
  """
  # decoder outputs raw floats, so threshold to get predicted bits
  predicted_bits = (decoded_message > 0).float()

  # compare every entry in D x H x W across the batch
  # wrong = 1 where they differ, 0 where they match
  wrong = (predicted_bits != message).float()

  # p = total wrong bits / total bits (single scalar across entire batch)
  p = wrong.mean().item()

  # RS-BPP formula: clamp to 0 if decoder is worse than random (p > 0.5)
  return max(0.0, D * (1 - 2 * p))

def psnr(cover_image, stego_image):
  """
  Inputs:
  - cover_image: N x 3 x H x W (normalized to [-1, 1])
  - stego_image: N x 3 x H x W (normalized to [-1, 1])

  Returns: scalar average PSNR in dB across the batch
  """
  # pixel range is 2.0 for images in [-1, 1]
  max_val = 2.0
  mse = F.mse_loss(stego_image, cover_image)
  return (10 * torch.log10(max_val ** 2 / mse)).item()

def ssim(cover_image, stego_image):
  """
  Inputs:
  - cover_image: N x 3 x H x W (normalized to [-1, 1])
  - stego_image: N x 3 x H x W (normalized to [-1, 1])

  Returns: scalar SSIM value, closer to 1 is better
  """
  device = cover_image.device
  metric = StructuralSimilarityIndexMeasure(data_range=2.0).to(device)
  return metric(stego_image, cover_image).item()

def evaluate(encoder, decoder, dataloader, D, device="cpu"):
  """
  Inputs:
  - encoder: trained encoder model
  - decoder: trained decoder model
  - dataloader: DataLoader over the validation set
  - D: bits per pixel (must match what encoder/decoder were trained with)
  - device: "cpu" or "cuda"

  Returns: dict with average RS-BPP, PSNR, SSIM across the validation set
  """

  #Switch the models from training mode to evaluation mode.
  #BatchNorm behaves differently during evaluation (uses running mean and variance)
  encoder.eval()
  decoder.eval()

  total_rs_bpp = 0.0
  total_psnr = 0.0
  total_ssim = 0.0
  num_batches = 0

  with torch.no_grad():
    for cover_image in dataloader: #cover_image is shape (N, 3, H, W)
      cover_image = cover_image.to(device)
      N, _, H, W = cover_image.shape

      # generate random binary message for this batch
      message = torch.randint(0, 2, (N, D, H, W), dtype=torch.float, device=device)

      # encode and decode
      stego_image = encoder(cover_image, message)
      decoded_message = decoder(stego_image)

      # accumulate metrics
      total_rs_bpp += rs_bpp(D, message, decoded_message)
      total_psnr += psnr(cover_image, stego_image)
      total_ssim += ssim(cover_image, stego_image)
      num_batches += 1

  return {
    "RS-BPP": total_rs_bpp / num_batches,
    "PSNR": total_psnr / num_batches,
    "SSIM": total_ssim / num_batches,
  }

Encoders

In [13]:
import torch
from torch import nn

class BasicEncoder(nn.Module):
    def __init__(self, D:int):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 32, 3,1,"same"),
            nn.LeakyReLU(inplace=True),
            nn.BatchNorm2d(32)
        )

        self.conv2 = nn.Sequential(
            nn.Conv2d(32+D, 32, 3,1,"same"),
            nn.LeakyReLU(inplace=True),
            nn.BatchNorm2d(32),

            nn.Conv2d(32, 32, 3,1,"same"),
            nn.LeakyReLU(inplace=True),
            nn.BatchNorm2d(32)
        )
        # omit leaky relu and batch norm for last block
        self.conv3 = nn.Conv2d(32,3,3,1,"same")

    def forward(self, cover_image, message):
        """
        Parameters
        ----------
        cover_image : N x 3 x H x W Tensor
            Cover image C to input to Encoder network, H x W is size of image and 3 RGB channels. N is batch size
        message: {0, 1} N x D x H x W Tensor
            Binary data tensor, secret message M. D is the number of bits to hide in each pixel of cover image.

        Returns
        -------
        output: N x 3 x H x W Tensor
            Output encoded steganographic image S
        """
        x = self.conv1(cover_image)
        x = torch.cat([x, message], dim = 1) # becomes N x 32 + D x H x W
        # pass concatenated input into the next two blocks sequentially
        x = self.conv2(x)
        x = self.conv3(x)
        x = torch.tanh(x) # get back to expected image output range [-1,1] since this predicts entire image
        return x


class ResidualEncoder(nn.Module):
    # def forward(self, cover_image, message):
    #     x = super().forward(cover_image,message)
    #     return cover_image + x

    def __init__(self, D:int):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 32, 3,1,"same"),
            nn.LeakyReLU(inplace=True),
            nn.BatchNorm2d(32)
        )

        self.conv2 = nn.Sequential(
            nn.Conv2d(32+D, 32, 3,1,"same"),
            nn.LeakyReLU(inplace=True),
            nn.BatchNorm2d(32),

            nn.Conv2d(32, 32, 3,1,"same"),
            nn.LeakyReLU(inplace=True),
            nn.BatchNorm2d(32)
        )
        # omit leaky relu and batch norm for last block
        self.conv3 = nn.Conv2d(32,3,3,1,"same")

    def forward(self, cover_image, message):
        """
        Parameters
        ----------
        cover_image : N x 3 x H x W Tensor
            Cover image C to input to Encoder network, H x W is size of image and 3 RGB channels. N is batch size
        message: {0, 1} N x D x H x W Tensor
            Binary data tensor, secret message M. D is the number of bits to hide in each pixel of cover image.

        Returns
        -------
        output: N x 3 x H x W Tensor
            Output encoded steganographic image S
        """
        x = self.conv1(cover_image)
        x = torch.cat([x, message], dim = 1) # becomes N x 32 + D x H x W
        # pass concatenated input into the next two blocks sequentially
        x = self.conv2(x)
        x = self.conv3(x)
        # no Tanh since it now predicts a residual from the input image
        return cover_image + x

class DenseEncoder(nn.Module):
    # def forward(self, cover_image, message):
    #     x = super().forward(cover_image,message)
    #     return cover_image + x

    def __init__(self, D:int):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 32, 3,1,"same"),
            nn.LeakyReLU(inplace=True),
            nn.BatchNorm2d(32)
        )

        self.conv2 = nn.Sequential(
            nn.Conv2d(32+D, 32, 3,1,"same"),
            nn.LeakyReLU(inplace=True),
            nn.BatchNorm2d(32)
        )
        self.conv3 = nn.Sequential(
            nn.Conv2d(32*2 + D, 32, 3,1,"same"),
            nn.LeakyReLU(inplace=True),
            nn.BatchNorm2d(32)
        )
        # omit leaky relu and batch norm for last block
        self.conv4 = nn.Conv2d(32*3 + D,3,3,1,"same")

    def forward(self, cover_image, message):
        """
        Parameters
        ----------
        cover_image : N x 3 x H x W Tensor
            Cover image C to input to Encoder network, H x W is size of image and 3 RGB channels. N is batch size
        message: {0, 1} N x D x H x W Tensor
            Binary data tensor, secret message M. D is the number of bits to hide in each pixel of cover image.

        Returns
        -------
        output: N x 3 x H x W Tensor
            Output encoded steganographic image S
        """
        # list of intermediate outputs
        xs = []
        x = self.conv1(cover_image)
        xs.append(x)
        x = torch.cat(xs + [message], dim = 1)
        x = self.conv2(x)
        xs.append(x)
        x = torch.cat(xs + [message], dim=1)
        x = self.conv3(x)
        xs.append(x)
        x = torch.cat(xs + [message], dim=1)
        x = self.conv4(x)
        # no Tanh since it now predicts a residual from the input image
        return cover_image + x

Decoder

In [14]:
import torch
import torch.nn as nn

class Decoder(nn.Module):
    """
    Spec: See Section 3.2.2 (Equation 6)
    Input Size: (N, 3, H, W)
    Output Size: (N, D, H, W)

    Experiments: Ablate over D={1, 3, 6}
    """

    def __init__(self, D, hidden_dim=32):
        super(Decoder, self).__init__()
        # conv_D-->D' blocks: (page 3, 4 in the paper)
        # (1) Conv2d with in_channel D, out_channel D', kernel size 3, stride 1, padding same (so padding = 1)
        # (2) LeakyRelU activation
        # (3) BatchNormalization
        # omit activation and batch norm if convolution block is last block in network

        self.hidden_dim = hidden_dim
        # cat operation: concat along the depth axis (the channel axis)
        self.a = nn.Conv2d(3, self.hidden_dim, 3, 1, 1)
        self.b = nn.Conv2d(self.hidden_dim, self.hidden_dim, 3, 1, 1)
        self.c = nn.Conv2d(2 * self.hidden_dim, self.hidden_dim, 3, 1, 1) # 64 because we concat a and b which each have 32 channels
        self.d = nn.Conv2d(3 * self.hidden_dim, D, 3, 1, 1) # output channels is D, the hidden_dim
        self.leakyrelu = nn.LeakyReLU(inplace=True)
        self.batchnorm = nn.BatchNorm2d(self.hidden_dim) # a, b, c, d have same hidden size (output channel size)


    def forward(self, x):
        a_x = self.batchnorm(self.leakyrelu(self.a(x)))
        b_x = self.batchnorm(self.leakyrelu(self.b(a_x)))
        c_x = self.batchnorm(self.leakyrelu(self.c(torch.concat([a_x, b_x], dim=1)))) # concat along the chanel dim
        d_x = self.d(torch.concat([a_x, b_x, c_x], dim=1))

        return d_x

Training

In [15]:
import torch
import torch.nn.functional as F
from torch.nn.utils import clip_grad_norm_
import argparse

class Trainer:
    """
    This training class jointly optimizes three networks:
    - Encoder: hides a binary message inside an image
    - Decoder: recovers the hidden message
    - Critic: distinguishes real vs generated images (Wasserstein GAN)

    For each batch:
    1. Sample a random binary message M ~ Ber(0.5)
    2. Update the critic using Wasserstein loss:
    Lc = C(real) - C(fake)
    3. Update encoder + decoder using:
    L = Ld + Ls + Lr
    where:
        Ld: decoding loss (binary cross entropy)
        Ls: similarity loss (MSE between cover and stego image)
        Lr: realism loss from critic

    After each epoch, evaluate using:
        RS-BPP (message capacity)
        PSNR (pixel-level distortion)
        SSIM (perceptual similarity)
    """
    def __init__(self, encoder, decoder, critic, D, device):
        self.encoder = encoder.to(device)
        self.decoder = decoder.to(device)
        self.critic = critic.to(device)

        self.device = device
        self.D = D

        self.enc_dec_opt = torch.optim.Adam(
            list(self.encoder.parameters()) + list(self.decoder.parameters()),
            lr=1e-4,
        )
        self.critic_opt = torch.optim.Adam(self.critic.parameters(), lr=1e-4)

        self.grad_clip = 0.25
        self.critic_clip = 0.1

    def sample_message(self, N, H, W):
        """
        Inputs:
        - N: batch size
        - D: bits per pixel (data depth)
        - H, W: spatial dimensions
        - device: torch device (cpu or cuda)

        Returns:
        - message: N x D x H x W tensor of binary values {0,1}

        Description:
        - Generates a random binary message for each image in the batch
        - Each pixel stores D bits
        - Values are sampled from a Bernoulli(0.5) distribution
        """
        return torch.randint(0, 2, (N, self.D, H, W), device=self.device).float()

    def similarity_loss(self, cover, generated):
        """
        Inputs:
        - cover: N x 3 x H x W original image
        - generated: N x 3 x H x W stego image

        Returns:
        - scalar similarity loss

        Description:
        - Computes normalized mean squared error between cover and stego images
        - Matches paper formulation:
            Ls = (1 / (3 * H * W)) * ||cover - generated||^2
        - Encourages minimal visual distortion
        """
        _, _, H, W = cover.shape
        return ((cover - generated) ** 2).sum(dim=(1, 2, 3)).mean() / (3 * H * W)

    def train_epoch(self, loader, epoch):

        self.encoder.train()
        self.decoder.train()
        self.critic.train()

        total = {"Lc": 0.0, "Ld": 0.0, "Ls": 0.0, "Lr": 0.0, "Acc": 0.0}
        steps = 0

        tqdm_bar = tqdm(loader, desc=f"Epoch: {epoch}", leave=True)
        for cover in tqdm_bar:
            cover = cover.to(self.device)
            N, _, H, W = cover.shape

            # sample message to hide for this batch
            M = self.sample_message(N, H, W)

            # Critic
            with torch.no_grad():
                fake = self.encoder(cover, M)

            # Critic Update (Wasserstein GAN + Gradient clipping (stability) + Weight clipping to enforce Lipschitz constraint)
            real_score = self.critic(cover).mean()
            fake_score = self.critic(fake).mean()
            Lc = real_score - fake_score

            self.critic_opt.zero_grad()
            Lc.backward()
            clip_grad_norm_(self.critic.parameters(), self.grad_clip)
            self.critic_opt.step()

            for p in self.critic.parameters():
                p.data.clamp_(-self.critic_clip, self.critic_clip)

            # Encoder-Decoder Loss & Updates
            fake = self.encoder(cover, M)
            decoded = self.decoder(fake)

            # Ld (decoding loss): Binary cross entropy between decoded message and original message
            Ld = F.binary_cross_entropy_with_logits(decoded, M)
            # Ls (similarity loss): MSE between cover and stego image
            Ls = self.similarity_loss(cover, fake)
            # Lr (realness loss): Critic score of generated image
            Lr = self.critic(fake).mean()

            loss = Ld + Ls + Lr

            self.enc_dec_opt.zero_grad()
            loss.backward()
            clip_grad_norm_(
                list(self.encoder.parameters()) + list(self.decoder.parameters()),
                self.grad_clip,
            )
            self.enc_dec_opt.step()

            with torch.no_grad():
                acc = ((decoded >= 0) == (M >= 0.5)).float().mean() # Measures fraction of correctly recovered bits

            total["Lc"] += Lc.item()
            total["Ld"] += Ld.item()
            total["Ls"] += Ls.item()
            total["Lr"] += Lr.item()
            total["Acc"] += acc.item()
            steps += 1

            wandb.log({"train/Lc": Lc.item(), "train/Ld": Ld.item(),
                "train/Ls": Ls.item(), "train/Lr": Lr.item(), "train/Acc": acc.item()})
            tqdm_bar.set_postfix({"Lc": f"{Lc.item():.3f}", "Ld": f"{Ld.item():.3f}", "Acc": f"{acc.item():.3f}"})

        return {k: v / steps for k, v in total.items()}

    def train(self, train_loader, val_loader, epochs):
        for epoch in range(epochs):
            print(f"\nEpoch {epoch+1}")

            train_metrics = self.train_epoch(train_loader, epoch+1)

            print(
                f"Lc: {train_metrics['Lc']:.4f} | "
                f"Ld: {train_metrics['Ld']:.4f} | "
                f"Ls: {train_metrics['Ls']:.6f} | "
                f"Lr: {train_metrics['Lr']:.4f} | "
                f"Acc: {train_metrics['Acc']:.4f}"
            )

            # Evaluation metrics, see evaluate.py for details
            eval_metrics = evaluate(self.encoder, self.decoder, val_loader, self.D, device=self.device)

            print(
                f"RS-BPP: {eval_metrics['RS-BPP']:.4f} | "
                f"PSNR: {eval_metrics['PSNR']:.2f} | "
                f"SSIM: {eval_metrics['SSIM']:.4f}"
            )

            wandb.log({"eval/RS-BPP": eval_metrics["RS-BPP"],
                "eval/PSNR": eval_metrics["PSNR"],
                "eval/SSIM": eval_metrics["SSIM"],
                "epoch": epoch + 1})



Critic

In [16]:
import torch
import torch.nn as nn

class Critic(nn.Module):
    """
    Spec: See Section 3.2.3 (Equation 7)
    Input Size: (N, 3, H, W)
    Output Size: (N, 1, H, W)
    """

    def __init__(self, hidden_dim=32):
        super(Critic, self).__init__()
        # (same as decoder)
        # conv_D-->D' blocks: (page 3, 4 in the paper)
        # (1) Conv2d with in_channel D, out_channel D', kernel size 3, stride 1, padding same (so padding = 1)
        # (2) LeakyRelU activation
        # (3) BatchNormalization
        # omit activation and batch norm if convolution block is last block in network

        self.hidden_dim = hidden_dim
        # cat operation: concat along the depth axis (the channel axis)
        self.a = nn.Conv2d(3, self.hidden_dim, 3, 1, 1) # Conv3->32
        self.b = nn.Conv2d(self.hidden_dim, self.hidden_dim, 3, 1, 1) # Conv32->32
        self.c = nn.Conv2d(self.hidden_dim, self.hidden_dim, 3, 1, 1) # Conv32->32
        self.d = nn.Conv2d(self.hidden_dim, 1, 3, 1, 1) # Conv32->1

        self.leakyrelu = nn.LeakyReLU(inplace=True)
        self.batchnorm = nn.BatchNorm2d(self.hidden_dim)


    def forward(self, x):
        a_x = self.batchnorm(self.leakyrelu(self.a(x)))
        b_x = self.batchnorm(self.leakyrelu(self.b(a_x)))
        c_x = self.batchnorm(self.leakyrelu(self.c(b_x)))
        d_x = self.d(c_x) # N x 1 x H x W
        d_x = d_x.mean(dim=[2, 3]).squeeze(1) # find single mean for each channel (mean across H x W dimension --> N x 1 --> (N, ) as a result of squeeze)

        return d_x # dimension: (N, )

Configs

In [ ]:
# Config sweep

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Paper trains: 3 architectures x 6 data depths = 18 configs
# lr=1e-4, grad_clip=0.25, critic_clip=0.1, epochs=32 (all fixed in paper)
EPOCHS   = 32
LR       = 1e-4

encoder_variants = {
    "basic":    BasicEncoder,
    "residual": ResidualEncoder,
    "dense":    DenseEncoder,
}

configs = [
    {"arch": arch_name, "D": D, "label": f"{arch_name}_D{D}"}
    for arch_name in ["basic", "residual", "dense"]
    for D in range(1, 7)      # D ∈ {1, 2, 3, 4, 5, 6}
]

all_results = {}

for cfg in configs:
    print(f"\n{'='*60}")
    print(f"Running config: {cfg['label']}  (arch={cfg['arch']}, D={cfg['D']})")
    print('='*60)

    EncoderClass = encoder_variants[cfg["arch"]]
    encoder = EncoderClass(D=cfg["D"])
    decoder = Decoder(D=cfg["D"])
    critic  = Critic()

    trainer = Trainer(encoder, decoder, critic, D=cfg["D"], device=device)
    # lr, grad_clip=0.25, critic_clip=0.1 are already set to paper defaults in Trainer

    wandb.init(
        project="steganogan_metrics",
        name=cfg["label"],
        config={"arch": cfg["arch"], "D": cfg["D"], "epochs": EPOCHS, "lr": LR},
    )

    trainer.train(train_loader, valid_loader, epochs=EPOCHS)

    wandb.finish()

    final = evaluate(encoder, decoder, valid_loader, cfg["D"], device=device)
    all_results[cfg["label"]] = {**final, "arch": cfg["arch"], "D": cfg["D"]}
    print(f"\n[{cfg['label']}] RS-BPP: {final['RS-BPP']:.4f} | "
          f"PSNR: {final['PSNR']:.2f} dB | SSIM: {final['SSIM']:.4f}")

# Summary table (based on paper Table 1)
print(f"\n{'='*72}")
print(f"{'D':<4} {'Arch':<12} {'RS-BPP':>8} {'PSNR':>8} {'SSIM':>8}")
print('-'*72)
for D in range(1, 7):
    for arch in ["basic", "residual", "dense"]:
        label = f"{arch}_D{D}"
        m = all_results[label]
        print(f"D={D}  {arch:<12} {m['RS-BPP']:>8.4f} {m['PSNR']:>8.2f} {m['SSIM']:>8.4f}")
    print()


Running config: basic_D1  (arch=basic, D=1)


train/Acc,▁▁▁▁▁▁▁▂▂▂▃▃▃▃▃▃▄▄▄▄▅▅▄▅▅▅▆▆▆▆▇▇▇▇▇▇▇▇██
train/Lc,█▄▄▅▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▃▃▃▃▃▃▃▃▃▃▃▃▃▂▃▃▂▁
train/Ld,██▇▇▇▇▇▇▇▇▇▆▆▆▆▆▆▆▆▆▅▆▅▅▅▄▄▄▅▄▃▃▃▃▃▂▂▂▁▁
train/Lr,▁▁▁▁▁▂▂▂▁▂▁▂▁▁▁▁▁▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▄▆▆▆▇█
train/Ls,█▇▇▅▄▄▃▃▄▄▃▅▃▂▂▂▂▂▂▂▂▂▂▁▂▂▂▁▂▁▁▁▁▁▂▁▁▁▁▂
train/Acc,0.74562
train/Lc,-0.00148
train/Ld,0.58042
train/Lr,0.01797
train/Ls,0.11246



Epoch 1


Epoch: 1: 100%|██████████| 50/50 [02:28<00:00,  2.96s/it, Lc=-0.001, Ld=0.594, Acc=0.704]


Lc: -0.0004 | Ld: 0.6564 | Ls: 0.118222 | Lr: -0.0103 | Acc: 0.6253
RS-BPP: 0.3115 | PSNR: 18.47 | SSIM: 0.3795

Epoch 2


Epoch: 2: 100%|██████████| 50/50 [02:25<00:00,  2.91s/it, Lc=-0.004, Ld=0.284, Acc=0.913]


Lc: -0.0019 | Ld: 0.4347 | Ls: 0.061459 | Lr: -0.0075 | Acc: 0.8230
RS-BPP: 0.6699 | PSNR: 18.57 | SSIM: 0.3158

Epoch 3


Epoch: 3:  46%|████▌     | 23/50 [01:07<01:15,  2.81s/it, Lc=-0.004, Ld=0.266, Acc=0.904]